In [2]:
from copy import deepcopy

In [3]:
def is_dominating_set(solution_nodes, graph):
    all_nodes = set(graph.keys())
    for node in all_nodes:
        is_dominated = False
        if node in solution_nodes:
            is_dominated = True
        else:
            for neighbor in graph[node]:
                if neighbor in solution_nodes:
                    is_dominated = True
                    break
        if not is_dominated:
            return False
    return True

In [4]:
def initialize(graph):
    return set(graph.keys())

In [5]:
def calculate_solution_value(solution_nodes):
    return len(solution_nodes)

In [6]:
def generate_neighbors(solution, graph):
    neighbors = []

    # Removal
    for node in solution:
        new_solution = deepcopy(solution)
        new_solution.remove(node)
        if is_dominating_set(new_solution, graph):
            neighbors.append((new_solution, ('removal', node)))

    # Swap
    non_solution = set(graph.keys()).difference(solution)
    for node_out in solution:
        for node_in in non_solution:
            new_solution = deepcopy(solution)
            new_solution.remove(node_out)
            new_solution.add(node_in)
            if is_dominating_set(new_solution, graph):
                neighbors.append((new_solution, ('swap', node_out, node_in)))

    return neighbors

In [7]:
def tabu_search(graph, max_iters, tabu_tenure):
    current_solution = initialize(graph)
    best_solution = deepcopy(current_solution)
    best_value = calculate_solution_value(best_solution)

    tabu_list = {} # key: move type, value: iteration up to which the move is taboo
    long_term_memory = {}

    for iteration in range(max_iters):
        expired_moves = [move for move, expiry in tabu_list.items() if expiry <= iteration]
        for move in expired_moves:
            del tabu_list[move]

        neighbors = generate_neighbors(current_solution, graph)
        candidate_moves = []

        for sol, move in neighbors:
            sol_value = calculate_solution_value(sol)
            is_tabu = move in tabu_list

            if is_tabu and sol_value >= best_value:
                continue

            candidate_moves.append((sol, move, sol_value))

        if not candidate_moves:
            break

        candidate_moves.sort(key=lambda x: (x[2], long_term_memory.get(x[1], 0)))
        chosen_solution, chosen_move, chosen_value = candidate_moves[0]

        current_solution = deepcopy(chosen_solution)

        if chosen_value < best_value:
            best_solution = deepcopy(current_solution)
            best_value = chosen_value

        tabu_list[chosen_move] = iteration + tabu_tenure
        long_term_memory[chosen_move] = long_term_memory.get(chosen_move, 0) + 1

    return best_solution, best_value

In [8]:
if __name__ == '__main__':
    graph = {
        1: [2, 27],
        2: [1, 30, 7],
        3: [30, 24, 25],
        4: [5, 19],
        5: [4, 8],
        6: [31, 15, 16],
        7: [2, 29, 17, 32],
        8: [5, 9, 17, 19],
        9: [8, 10],
        10: [9, 17],
        11: [12, 13, 24],
        12: [11, 14, 26],
        13: [11, 23, 28],
        14: [12, 29, 18],
        15: [6, 20, 21, 16],
        16: [6, 15, 31, 32],
        17: [7, 8, 10, 27],
        18: [14, 29, 19],
        19: [4, 8, 18],
        20: [15, 21],
        21: [15, 20, 22],
        22: [21, 29, 26],
        23: [13, 28, 24],
        24: [3, 11, 23, 25],
        25: [3, 24],
        26: [12, 22, 29],
        27: [1, 17],
        28: [13, 23],
        29: [7, 14, 18, 22, 26, 32],
        30: [2, 3, 31, 32],
        31: [6, 16, 30],
        32: [7, 16, 29, 30]
    }

    best_solution, best_value = tabu_search(graph, max_iters=100, tabu_tenure=3)
    print("Best solution:", best_solution)
    print("Best value:", best_value)

Best solution: {4, 9, 15, 24, 26, 27, 28, 29, 30}
Best value: 9
